# Spark + Apache Iceberg on Snowflake: StreamCo churn demo

Run your existing **PySpark** code on Snowflake with **Snowpark Connect for Spark (SCOS)**, against open **Apache Iceberg** tables.

**The story:** StreamCo, a streaming company, wants to know which subscribers are about to cancel. Their pipeline is PySpark. We run it unchanged on Snowflake and pick up the new Iceberg features along the way.

| Section | Feature |
|---|---|
| 1. Bronze | Spark file readers on a stage, DataFrameWriterV2 |
| 2. Silver | WriterV2 `partitionedBy(days())`, table properties, `overwritePartitions()` |
| 3. Gold + tags | Iceberg tags to bookmark the model-training snapshot |
| 4. Incremental CDC | Incremental reads between snapshots, `.changes` changelog, `overwrite(condition)` |
| 5. Time travel | By tag, snapshot ID and timestamp |
| 6. Write-Audit-Publish | Spark WAP on Iceberg branches + `fast_forward` |
| 7. Lineage | `.snapshots`, `.refs`, `system.ancestors_of` |

Sections 1-7 use Snowflake-managed Iceberg (Horizon catalog, Snowflake storage) and need no cloud setup.

## Before you start

1. Run **`setup.sql`** first (as ACCOUNTADMIN). It creates the role, warehouse, database, schemas and stage.
2. In Snowsight, open **Projects > Workspaces**, upload this `.ipynb`, and open it.
3. Connect the notebook to a service. Under **Artifact Repository**, pick `snowflake.snowpark.pypi_shared_repository` so the next cell can install the latest Snowpark Connect.
4. Run the notebook as **SCOS_CHURN_ROLE**.
5. Spark WAP (section 6) is in preview. If your account doesn't have it yet, section 6 fails; everything else still runs.

Edit the config cell below, then **Run all**.

In [ ]:
# Install the Snowpark Connect version with all Iceberg features (>= 1.44.0).
# Restart the kernel once after this finishes (Connect > Restart kernel), then continue.
!pip install -q "snowpark-connect[iceberg]>=1.44.0"

## 0. Config

In [ ]:
DB = "CHURN_DEMO"            # database created by setup.sql
WAREHOUSE = "SCOS_CHURN_WH"     # warehouse to run on
N_CUSTOMERS = 50_000         # synthetic customers
DAYS = 60                    # days of viewing history

## Session setup

One Snowpark session drives everything: Spark (through Snowpark Connect) and the few Snowflake SQL statements Spark can't express. Inside a Snowflake notebook it uses the notebook's own session; run locally, it uses a named connection.

In [ ]:
import os

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Connected via Snowflake notebook")
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "default")).create()
    print("Connected locally")

session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
try:
    session.sql(f"USE SCHEMA {DB}.BRONZE").collect()  # SCOS needs a current database + schema
    session.sql(f"DESCRIBE STAGE {DB}.BRONZE.RAW_LANDING").collect()
except Exception as e:
    raise RuntimeError(f"{DB} is not set up yet - run setup.sql first") from e
print(session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA()").collect()[0])

In [ ]:
import logging
from snowflake import snowpark_connect

logging.getLogger("snowflake_connect_server").setLevel(logging.WARNING)  # keep cell output readable

# Start Snowpark Connect on the same Snowpark session, then get a Spark session
snowpark_connect.start_session(snowpark_session=session)
spark = snowpark_connect.get_session()
# Iceberg SQL extensions: needed for branches, tags and VERSION/TIMESTAMP AS OF
spark.conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

# Spark Connect functions (the classic pyspark.sql.functions would look for a local JVM)
from pyspark.sql.connect import functions as F
import time


def latest_snapshot(table):
    return spark.sql(f"SELECT snapshot_id FROM {table}.snapshots ORDER BY committed_at DESC LIMIT 1").collect()[0][0]


def with_retry(fn, attempts=5, wait_s=15):
    # A write right after table creation can hit Snowflake's background Iceberg maintenance (error 093924)
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:
            if "background maintenance" not in str(e) or i == attempts - 1:
                raise
            print(f"Table busy with background maintenance, retrying in {wait_s}s...")
            time.sleep(wait_s)


from importlib.metadata import version
print("snowpark-connect", version("snowpark-connect"))
spark.sql("SELECT 1 AS ok").show()

## Generate synthetic StreamCo data

Creates raw files on the stage `@RAW_LANDING`, the way an upstream system would drop them:

| Folder | Format | Content |
|---|---|---|
| `initial/subscriptions/` | CSV | one row per customer: plan, monthly revenue, start/cancel dates |
| `initial/usage_events/` | JSON lines | viewing sessions over 60 days |
| `initial/support_tickets/` | CSV | tickets with a sentiment score |
| `increment/usage_events/` | JSON lines | one new day of viewing (for section 4) |
| `wap_bad/`, `wap_good/` | CSV | incoming subscription batches (for section 6) |

Customers with a high hidden "risk" watch less over time, file more tickets, and are more likely to cancel, so the churn score in section 3 has real signal.

In [ ]:
import csv
import io
import json
import random
from datetime import date, datetime, timedelta

random.seed(42)
START = date(2026, 8, 1)
STAGE = f"@{DB}.BRONZE.RAW_LANDING"
PLANS = [("basic", 7.99), ("standard", 15.49), ("premium", 22.99)]
DEVICES = ["tv", "mobile", "web", "tablet"]
SUB_HEADER = ["customer_id", "plan_tier", "mrr", "start_date", "cancel_date", "country"]


def risk(cid):
    return random.Random(cid).random()  # stable per customer


def put(path, text):
    session.file.put_stream(io.BytesIO(text.encode()), f"{STAGE}/{path}", auto_compress=False, overwrite=True)


def to_csv(header, rows):
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(header)
    w.writerows(rows)
    return buf.getvalue()


def usage_day(d):
    day = START + timedelta(days=d)
    rows = []
    for cid in random.sample(range(1, N_CUSTOMERS + 1), int(N_CUSTOMERS * 0.16)):
        r = risk(cid)
        base = 120 * (1 - r) * (1 - 0.6 * r * d / DAYS)  # risky users fade out
        ts = datetime.combine(day, datetime.min.time()) + timedelta(seconds=random.randint(0, 86_399))
        rows.append(json.dumps({"customer_id": f"C{cid:06d}", "event_ts": ts.isoformat(sep=" "),
                                "minutes_watched": max(1, int(random.gauss(base, 25))),
                                "device": random.choice(DEVICES)}))
    return rows


# Subscriptions
subs = []
for cid in range(1, N_CUSTOMERS + 1):
    plan, mrr = random.choice(PLANS)
    start = START - timedelta(days=random.randint(30, 900))
    cancel = (START + timedelta(days=random.randint(10, DAYS - 1))).isoformat() if risk(cid) > 0.82 else ""
    subs.append([f"C{cid:06d}", plan, mrr, start.isoformat(), cancel, random.choice(["US", "CA", "UK", "DE", "IN"])])
put("initial/subscriptions/subscriptions.csv", to_csv(SUB_HEADER, subs))

# Usage events, 10 days per file
for f in range(0, DAYS, 10):
    lines = [l for d in range(f, min(f + 10, DAYS)) for l in usage_day(d)]
    put(f"initial/usage_events/usage_{f // 10:03d}.json", "\n".join(lines))
put("increment/usage_events/usage_000.json", "\n".join(usage_day(DAYS)))

# Support tickets
tickets = []
for _ in range(int(N_CUSTOMERS * 0.6)):
    cid = random.randint(1, N_CUSTOMERS)
    r = risk(cid)
    if random.random() > 0.3 + r * 0.7:
        continue
    ts = datetime.combine(START + timedelta(days=random.randint(0, DAYS - 1)), datetime.min.time())
    tickets.append([f"C{cid:06d}", ts.isoformat(sep=" "), random.choice(["billing", "playback", "account", "content"]),
                    round(max(-1, min(1, random.gauss(0.4 - r, 0.3))), 2)])
put("initial/support_tickets/support_tickets.csv", to_csv(["customer_id", "ticket_ts", "category", "sentiment"], tickets))


# Two incoming batches for WAP: one broken billing export, one clean
def wap_batch(bad):
    rows = []
    for i in range(500):
        plan, mrr = random.choice(PLANS)
        cid = f"C{N_CUSTOMERS + i + 1:06d}"
        if bad and i % 7 == 0:
            cid = ""      # missing customer_id
        if bad and i % 11 == 0:
            mrr = -mrr    # negative revenue
        rows.append([cid, plan, mrr, (START + timedelta(days=DAYS)).isoformat(), "", "US"])
    return rows


put("wap_bad/subscriptions.csv", to_csv(SUB_HEADER, wap_batch(True)))
put("wap_good/subscriptions.csv", to_csv(SUB_HEADER, wap_batch(False)))

for r in session.sql(f"LIST {STAGE}").collect():
    print(f"{r['size']:>10,}  {r['name']}")

## 1. Bronze: raw files to Iceberg

Standard Spark readers (`spark.read.csv` / `.json`) read straight from the Snowflake stage. `writeTo(...).using("iceberg")` is **DataFrameWriterV2**; it lands each dataset as a Snowflake-managed Iceberg table.

In [ ]:
subs_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{STAGE}/initial/subscriptions/")
usage_raw = spark.read.json(f"{STAGE}/initial/usage_events/")
tickets_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{STAGE}/initial/support_tickets/")
subs_raw.printSchema()

for name, df in [("SUBSCRIPTIONS_RAW", subs_raw), ("USAGE_EVENTS_RAW", usage_raw), ("SUPPORT_TICKETS_RAW", tickets_raw)]:
    df.writeTo(f"{DB}.BRONZE.{name}").using("iceberg").createOrReplace()
    print(f"BRONZE.{name}: {spark.table(f'{DB}.BRONZE.{name}').count():,} rows")

## 2. Silver: clean, type and partition (DataFrameWriterV2)

- `partitionedBy(F.days("event_date"))` creates an Iceberg day-partitioned table.
- `tableProperty(...)` sets Iceberg table properties (comment, format version).
- `overwritePartitions()` replaces **only** the days present in the DataFrame, here a late-arriving fix for the last 3 days.
- Change tracking is turned on so section 4 can do incremental and changelog reads.

In [ ]:
subs = (spark.table(f"{DB}.BRONZE.SUBSCRIPTIONS_RAW")
        .select("customer_id", "plan_tier", F.col("mrr").cast("double").alias("mrr"),
                F.to_date("start_date").alias("start_date"), F.to_date("cancel_date").alias("cancel_date"), "country")
        .dropDuplicates(["customer_id"]))
(subs.writeTo(f"{DB}.SILVER.SUBSCRIPTIONS").using("iceberg")
     .tableProperty("comment", "StreamCo subscriptions - validated via WAP")
     .tableProperty("format-version", "2")
     .createOrReplace())

usage = (spark.table(f"{DB}.BRONZE.USAGE_EVENTS_RAW")
         .withColumn("event_ts", F.to_timestamp("event_ts"))
         .groupBy("customer_id", F.to_date("event_ts").alias("event_date"))
         .agg(F.sum("minutes_watched").alias("minutes_watched"), F.count("*").alias("sessions")))
(usage.writeTo(f"{DB}.SILVER.USAGE_DAILY").using("iceberg")
      .partitionedBy(F.days("event_date"))
      .tableProperty("comment", "Daily viewing per customer")
      .createOrReplace())

(spark.table(f"{DB}.BRONZE.SUPPORT_TICKETS_RAW").withColumn("ticket_ts", F.to_timestamp("ticket_ts"))
     .writeTo(f"{DB}.SILVER.SUPPORT_TICKETS").using("iceberg").createOrReplace())

# CHANGE_TRACKING has no Spark equivalent, so use Snowflake SQL
for t in ["SUBSCRIPTIONS", "USAGE_DAILY"]:
    session.sql(f"ALTER ICEBERG TABLE {DB}.SILVER.{t} SET CHANGE_TRACKING = TRUE").collect()

In [ ]:
# Late-arriving fix for the last 3 days: overwritePartitions() touches only those partitions
last_day = (START + timedelta(days=DAYS - 1)).isoformat()
fix = (usage.filter(F.col("event_date") >= F.date_sub(F.lit(last_day), 2))
            .withColumn("minutes_watched", F.col("minutes_watched") + 1))
with_retry(lambda: fix.writeTo(f"{DB}.SILVER.USAGE_DAILY").overwritePartitions())

spark.sql(f"SELECT partition, record_count FROM {DB}.SILVER.USAGE_DAILY.partitions ORDER BY partition DESC LIMIT 5").show(truncate=False)
for t in ["SUBSCRIPTIONS", "USAGE_DAILY", "SUPPORT_TICKETS"]:
    print(f"SILVER.{t}: {spark.table(f'{DB}.SILVER.{t}').count():,} rows")

## 3. Gold: churn features + an Iceberg tag

One row per customer with an explainable churn risk score (0-100): usage trend, support tickets and ticket sentiment. It stands in for a real model.

Then we **tag** this snapshot `model_train_q3`. A tag is an immutable pointer to one snapshot, so we can always get back to the exact data the model was trained on.

In [ ]:
AS_OF = (START + timedelta(days=DAYS)).isoformat()
GOLD = f"{DB}.GOLD.CHURN_FEATURES"


def churn_features(customer_filter=None):
    s = spark.table(f"{DB}.SILVER.SUBSCRIPTIONS")
    if customer_filter is not None:
        s = s.join(customer_filter, "customer_id", "left_semi")
    as_of = F.to_date(F.lit(AS_OF))
    u = spark.table(f"{DB}.SILVER.USAGE_DAILY").groupBy("customer_id").agg(
        F.sum(F.when(F.col("event_date") >= F.date_sub(as_of, 14), F.col("minutes_watched")).otherwise(0)).alias("mins_last_14d"),
        F.sum(F.when(F.col("event_date") < F.date_sub(as_of, 14), F.col("minutes_watched")).otherwise(0)).alias("mins_prior"),
        F.max("event_date").alias("last_active_date"))
    t = spark.table(f"{DB}.SILVER.SUPPORT_TICKETS").groupBy("customer_id").agg(
        F.count("*").alias("ticket_count"), F.round(F.avg("sentiment"), 2).alias("avg_sentiment"))
    f = (s.join(u, "customer_id", "left").join(t, "customer_id", "left")
         .fillna({"mins_last_14d": 0, "mins_prior": 0, "ticket_count": 0, "avg_sentiment": 0.0})
         .withColumn("tenure_days", F.datediff(as_of, "start_date"))
         .withColumn("usage_trend", F.round(F.col("mins_last_14d") / F.greatest(F.col("mins_prior") / 3.0, F.lit(1.0)), 2))
         .withColumn("churned", F.col("cancel_date").isNotNull()))
    score = (40 * (1 - F.least(F.col("usage_trend"), F.lit(1.0)))
             + 8 * F.least(F.col("ticket_count"), F.lit(5))
             + 20 * (F.col("avg_sentiment") < 0).cast("int"))
    return f.withColumn("churn_risk", F.round(F.least(score, F.lit(100.0)), 1)).select(
        "customer_id", "plan_tier", "mrr", "country", "tenure_days", "mins_last_14d", "usage_trend",
        "ticket_count", "avg_sentiment", "last_active_date", "churned", "churn_risk")


churn_features().writeTo(GOLD).using("iceberg").tableProperty("comment", "Churn model features").createOrReplace()
spark.table(GOLD).orderBy(F.col("churn_risk").desc()).show(5)
spark.sql(f"SELECT churned, count(*) customers, round(avg(churn_risk), 1) avg_risk FROM {GOLD} GROUP BY churned").show()

In [ ]:
# Bookmark the training snapshot.
# Note: "CREATE TAG ... RETAIN N DAYS" fails on SCOS 1.47, so we use the bare form.
spark.sql(f"ALTER TABLE {GOLD} CREATE TAG model_train_q3")
spark.sql(f"SELECT name, type, snapshot_id FROM {GOLD}.refs").show(truncate=False)

## 4. Incremental CDC: process only what changed

A new day of viewing data arrives. Instead of reprocessing everything:

1. **Incremental read:** `start-snapshot-id` / `end-snapshot-id` return only rows appended between two snapshots.
2. **Changelog:** the `.changes` table shows row-level inserts and deletes (including the `overwritePartitions` from section 2).
3. **Targeted rescore:** `writeTo(...).overwrite(condition)` replaces only the affected customers in the gold table.

In [ ]:
USAGE = f"{DB}.SILVER.USAGE_DAILY"
start_snap = latest_snapshot(USAGE)

new_day = (spark.read.json(f"{STAGE}/increment/usage_events/")
           .withColumn("event_ts", F.to_timestamp("event_ts"))
           .groupBy("customer_id", F.to_date("event_ts").alias("event_date"))
           .agg(F.sum("minutes_watched").alias("minutes_watched"), F.count("*").alias("sessions")))
new_day.writeTo(USAGE).append()
end_snap = latest_snapshot(USAGE)

delta = (spark.read.format("iceberg")
         .option("start-snapshot-id", start_snap)
         .option("end-snapshot-id", end_snap)
         .load(USAGE))
print(f"Full table: {spark.table(USAGE).count():,} rows | incremental read: {delta.count():,} rows")
delta.show(3)

In [ ]:
# Changelog: row-level inserts and deletes
spark.sql(f"SELECT _change_type, count(*) AS n FROM {USAGE}.changes GROUP BY _change_type").show()

In [ ]:
# Rescore only the customers that had new activity
affected = delta.select("customer_id").distinct()
ids = [r[0] for r in affected.collect()]
churn_features(customer_filter=affected).writeTo(GOLD).overwrite(F.col("customer_id").isin(ids))
print(f"Rescored {len(ids):,} of {spark.table(GOLD).count():,} customers")

## 5. Time travel: what changed since the model was trained?

Read the gold table three ways (by **tag**, **snapshot ID** and **timestamp**), then compare current scores with the training snapshot.

In [ ]:
snaps = spark.sql(f"SELECT snapshot_id, committed_at, operation FROM {GOLD}.snapshots ORDER BY committed_at")
snaps.show(truncate=False)
first_snap = snaps.collect()[0]["snapshot_id"]

by_tag = spark.read.format("iceberg").option("tag", "model_train_q3").load(GOLD)
by_snap = spark.read.format("iceberg").option("snapshot-id", first_snap).load(GOLD)
by_ts = spark.read.format("iceberg").option("as-of-timestamp", int(time.time() * 1000)).load(GOLD)
print(f"tag model_train_q3 : {by_tag.count():,} rows")
print(f"snapshot-id        : {by_snap.count():,} rows")
print(f"as-of-timestamp now: {by_ts.count():,} rows")

# Same thing in Spark SQL
spark.sql(f"SELECT count(*) AS n FROM {GOLD} VERSION AS OF 'model_train_q3'").show()

In [ ]:
now = spark.table(GOLD).select("customer_id", F.col("churn_risk").alias("risk_now"))
then = by_tag.select("customer_id", F.col("churn_risk").alias("risk_at_training"))
drift = (now.join(then, "customer_id")
         .withColumn("delta", F.round(F.col("risk_now") - F.col("risk_at_training"), 1))
         .filter("delta != 0"))
print(f"{drift.count():,} customers changed risk since model_train_q3")
drift.orderBy(F.abs("delta").desc()).show(5)

## 6. Write-Audit-Publish (WAP) on Iceberg branches

Bad data should never reach the table consumers read. With WAP:

1. **Write:** create a branch and land the batch there. `main` doesn't change.
2. **Audit:** run quality checks against the branch.
3. **Publish or reject:** if the checks pass, `fast_forward` moves `main` to the branch in one metadata-only commit. If they fail, drop the branch.

We run it twice: a broken billing export (missing IDs, negative revenue), then a clean batch.

> Read the branch with `spark.read.format("iceberg").option("branch", b).load(t)`. On SCOS 1.47 the `option("branch", b).table(t)` form silently returns `main`.

In [ ]:
SUBS = f"{DB}.SILVER.SUBSCRIPTIONS"


def load_batch(folder):
    return (spark.read.option("header", "true").option("inferSchema", "true").csv(f"{STAGE}/{folder}/")
            .select("customer_id", "plan_tier", F.col("mrr").cast("double").alias("mrr"),
                    F.to_date("start_date").alias("start_date"), F.to_date("cancel_date").alias("cancel_date"), "country"))


def audit(branch):
    df = spark.read.format("iceberg").option("branch", branch).load(SUBS)
    checks = {
        "null customer_id": df.filter("customer_id IS NULL OR customer_id = ''").count(),
        "negative mrr": df.filter("mrr < 0").count(),
        "duplicate customer_id": df.count() - df.select("customer_id").distinct().count(),
    }
    for name, bad in checks.items():
        print(f"  {'FAIL' if bad else 'PASS'}  {name}: {bad}")
    return all(v == 0 for v in checks.values())


def wap(folder):
    branch = f"audit_{folder}_{datetime.now():%H%M%S}"
    before = spark.table(SUBS).count()
    print(f"WRITE   {folder} -> branch {branch}")
    spark.sql(f"ALTER TABLE {SUBS} CREATE BRANCH {branch}")
    spark.conf.set("spark.wap.branch", branch)   # writes in this session go to the branch
    load_batch(folder).writeTo(SUBS).append()
    spark.conf.unset("spark.wap.branch")
    print(f"        main still has {spark.table(SUBS).count():,} rows")
    print(f"AUDIT   {branch}")
    if audit(branch):
        print("PUBLISH fast_forward main -> branch")
        spark.sql(f"CALL {DB}.system.fast_forward('{SUBS}', 'main', '{branch}')").collect()
    else:
        print("REJECT  drop the branch; main was never touched")
    spark.sql(f"ALTER TABLE {SUBS} DROP BRANCH {branch}")
    print(f"        main rows: {before:,} -> {spark.table(SUBS).count():,}\n")

In [ ]:
wap("wap_bad")

In [ ]:
wap("wap_good")

## 7. Lineage: prove what shipped

Iceberg metadata tables show every snapshot and every ref (branch or tag). The rejected batch and the published batch both appear as appends, but only the good one is an ancestor of `main`.

In [ ]:
spark.sql(f"SELECT snapshot_id, parent_id, committed_at, operation FROM {SUBS}.snapshots ORDER BY committed_at").show(truncate=False)
spark.sql(f"SELECT name, type, snapshot_id FROM {SUBS}.refs").show(truncate=False)
spark.sql(f"CALL {DB}.system.ancestors_of('{SUBS}')").show(truncate=False)
spark.sql(f"SELECT name, type, snapshot_id FROM {GOLD}.refs").show(truncate=False)

## Clean up

Uncomment to remove everything this notebook created.

In [ ]:
# session.sql(f"DROP DATABASE IF EXISTS {DB}").collect()